# Analysis of AML Model Predictions on MRD/MDS Patients

Comprehensive analysis including ring-wise occlusion attribution of nuclear morphology.

**Key analyses:**
1. Prediction score distributions by MRD group
2. Classification confidence analysis
3. Patient-level heterogeneity
4. Ring-wise occlusion attribution (8 concentric rings from lamina to center)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats, ndimage
from scipy.ndimage import distance_transform_edt
from skimage import filters, morphology
from skimage.measure import label as sk_label, regionprops
from PIL import Image
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# Set paths
PROJECT_ROOT = Path('/path/to/ainu_root/')
TEST_DATA_DIR = PROJECT_ROOT / 'results/occlusion_analysis/ainu_split3_model_best.pth.tar/ainu_split3_MRD_MDS'
IMAGE_DIR = PROJECT_ROOT / 'data/human_data'

# Output locations for exported stats CSVs and saved figures
OUTPUT_DIR = TEST_DATA_DIR / 'analysis_output'
STATS_DIR = OUTPUT_DIR / 'exported_data'
FIG_DIR = OUTPUT_DIR / 'figures'
STATS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Test data: {TEST_DATA_DIR}')
print(f'Test data exists: {TEST_DATA_DIR.exists()}')
print(f'Image dir exists: {IMAGE_DIR.exists()}')
print(f'Stats will be saved to: {STATS_DIR}')
print(f'Figures will be saved to: {FIG_DIR}')

# ── Shared ring-map segmentation, used identically for the occlusion attribution rings
# (Section 2) and the raw-image DNA-density rings (Section 5B) so both refer to the exact
# same anatomical rings (0 = boundary/lamina, n-1 = centre), anchored to the actual
# segmented cell in the raw DAPI image rather than to the attribution signal itself. ──
PAD = 5  # dark border width (pixels) added before thresholding, avoids edge (crop-boundary) artifacts

def extract_cell_mask_bg(gray: np.ndarray, sigma: float = 1.5, pad: int = PAD) -> np.ndarray:
    """Background-threshold cell mask. Exact replica of extract_cell_mask(mask_type='bg')
    from dapi_mask.ipynb / the Figure 6 occlusion-ring pipeline."""
    padded = np.pad(gray, pad, mode='constant', constant_values=0)
    smoothed = filters.gaussian(padded, sigma=sigma)
    thresh_bg = smoothed.max() * 0.2
    binary = smoothed > thresh_bg
    binary = ndimage.binary_fill_holes(binary)
    binary = morphology.remove_small_objects(binary, min_size=int(0.01 * binary.size))
    labeled = sk_label(binary)
    if labeled.max() == 0:
        return binary[pad:-pad, pad:-pad].astype(bool)
    largest = max(regionprops(labeled), key=lambda r: r.area).label
    mask = ndimage.binary_fill_holes(labeled == largest)
    return mask[pad:-pad, pad:-pad].astype(bool)

def ring_map(mask: np.ndarray, n_rings: int = 8, pad: int = PAD) -> np.ndarray:
    """Normalized-EDT ring labels: -1 = background, 0 = boundary, n-1 = centre.
    Matches occlusion_features()/ring_map() in the Figure 6 pipeline: linspace(0,1,n+1) bins,
    each shell restricted to the mask (background is never counted into ring 0)."""
    padded_mask = np.pad(mask, pad, mode='constant', constant_values=False)
    dist = ndimage.distance_transform_edt(padded_mask)[pad:-pad, pad:-pad]
    max_d = float(dist[mask].max()) if mask.any() else 1.0
    dist_n = dist / max_d
    bin_edges = np.linspace(0, 1, n_rings + 1)
    rmap = np.full(mask.shape, -1, dtype=np.int8)
    for i in range(n_rings):
        shell = mask & (dist_n >= bin_edges[i]) & (dist_n <= bin_edges[i + 1])
        rmap[shell] = i
    return rmap

## 1. Load and Merge Data

In [ ]:
# Load predictions
predictions_df = pd.read_csv(TEST_DATA_DIR / 'predictions.csv')
print(f'Predictions: {predictions_df.shape}')
print(f'Columns: {predictions_df.columns.tolist()}')

# Load metadata
metadata_df = pd.read_csv(TEST_DATA_DIR / 'MRD_MDS.csv', sep='\t')
print(f'\nMetadata: {metadata_df.shape}')
print(f'MRD values: {metadata_df["MRD"].value_counts(dropna=False)}')

In [ ]:
# Merge predictions with metadata
merged_df = predictions_df.merge(
    metadata_df[['File', 'MRD', 'patientID', 'disease_type']],
    left_on='image_name',
    right_on='File',
    how='left'
)

# Granular subgroup: Preleukemia disease type (MDS/CMML/CCUS) or MRD monitoring subtype
# (negative/unspecific/low-positive/positive) for de novo patients. AML _post-CMML is excluded
# (its name suggests these are already-transformed AML cases, not preleukemic).
def assign_cell_type(row):
    if row['disease_type'] == 'de novo':
        return row['MRD']
    if row['disease_type'] in ('MDS', 'CMML', 'CCUS'):
        return row['disease_type']
    return None

merged_df['cell_type'] = merged_df.apply(assign_cell_type, axis=1)

CELL_TYPE_ORDER = ['MDS', 'CMML', 'CCUS', 'negative', 'unspecific', 'low-positive', 'positive']
CELL_TYPE_COLORS = {
    'MDS': 'coral', 'CMML': 'goldenrod', 'CCUS': 'mediumseagreen', 'negative': 'mediumpurple',
    'unspecific': 'gray', 'low-positive': 'steelblue', 'positive': 'darkred',
}
PRELEUKEMIA_TYPES = ['MDS', 'CMML', 'CCUS']
MRD_TYPES = ['negative', 'unspecific', 'low-positive', 'positive']

print(f'Merged shape: {merged_df.shape}')
print(f'\ncell_type distribution (cells):')
print(merged_df['cell_type'].value_counts().reindex(CELL_TYPE_ORDER))

n_patients_per_type = merged_df.groupby('cell_type')['patientID'].nunique().reindex(CELL_TYPE_ORDER)
print(f'\ncell_type distribution (patients):')
print(n_patients_per_type)
print(f'\nExcluded (AML _post-CMML / other): {merged_df["cell_type"].isna().sum()} cells')

## 2. Extract Ring-wise Occlusion Attribution Features

In [ ]:
# Get all occlusion heatmap files
occlusion_files = sorted(glob.glob(str(TEST_DATA_DIR / '*occlusion_AML_baseline.npy')))
print(f'Found {len(occlusion_files)} occlusion heatmap files')

# Parameters
N_RINGS = 8

# Extract features - ring map now comes from the SAME raw-image segmentation
# (extract_cell_mask_bg + ring_map, defined in the setup cell) used for the DNA-density
# ring analysis in Section 5B, instead of thresholding the attribution heatmap itself.
# This anchors ring 0/lamina to the actual cell boundary in the raw DAPI image, not to
# wherever the attribution signal happens to be non-negligible.
print(f'\nExtracting ring-wise features (raw-image ring map)...\n')

ring_data = []
successful = 0
skipped_no_image = 0

for file_idx, npy_file in enumerate(occlusion_files):
    if (file_idx + 1) % 300 == 0:
        print(f'  {file_idx + 1}/{len(occlusion_files)} processed - {successful} extracted')

    try:
        # Extract clean image name from filename (occlusion files end in _positive/_preleukemia)
        basename = os.path.basename(npy_file)
        image_name = basename.replace('_occlusion_AML_baseline.npy', '')
        for suffix in ('_positive', '_preleukemia'):
            if image_name.endswith(suffix):
                image_name = image_name[:-len(suffix)]
                break

        img_path = IMAGE_DIR / f'{image_name}.png'
        if not img_path.exists():
            skipped_no_image += 1
            continue

        # Load occlusion heatmap (3 channels, 128x128), average across channels
        occl_heatmap = np.load(npy_file)
        occl_map = occl_heatmap.mean(axis=0)

        # Separate positive and negative attribution
        pos_attr = np.maximum(occl_map, 0)   # AML-pushing features
        neg_attr = np.abs(np.minimum(occl_map, 0))  # Control-pushing features

        # Raw-image cell mask + ring map, resized to the occlusion heatmap's own grid so
        # every pixel lines up 1:1 with occl_map (same coordinate frame the model saw).
        raw_img = Image.open(img_path).convert('L').resize(occl_map.shape[::-1], Image.BILINEAR)
        raw_arr = np.array(raw_img).astype(np.float64)

        mask = extract_cell_mask_bg(raw_arr)
        if mask.sum() < 10:
            continue

        rmap = ring_map(mask, n_rings=N_RINGS)

        # Extract per-ring statistics
        row_data = {'image_name': image_name}

        for ring_idx in range(N_RINGS):
            ring_mask = rmap == ring_idx

            if ring_mask.sum() > 0:
                pos_vals = pos_attr[ring_mask]
                neg_vals = neg_attr[ring_mask]

                row_data[f'attr_pos_ring{ring_idx:02d}_mean'] = float(pos_vals.mean())
                row_data[f'attr_pos_ring{ring_idx:02d}_std'] = float(pos_vals.std())
                row_data[f'attr_neg_ring{ring_idx:02d}_mean'] = float(neg_vals.mean())
                row_data[f'attr_neg_ring{ring_idx:02d}_std'] = float(neg_vals.std())
                row_data[f'attr_ring{ring_idx:02d}_mean'] = float(pos_vals.mean() - neg_vals.mean())

        ring_data.append(row_data)
        successful += 1

    except Exception as e:
        pass

print(f'\n✓ Extracted ring features from {successful} cells ({skipped_no_image} skipped: no matching raw image)')

In [ ]:
# Build ring_df from extracted data, then clean names and merge metadata
ring_df = pd.DataFrame(ring_data)
print(f'ring_df created: {ring_df.shape}')

# Clean image_name: occlusion filenames end in "_positive" or "_preleukemia" (verified against actual files on disk)
ring_df['image_name'] = ring_df['image_name'].str.replace(
    r'_(positive|preleukemia)$', '', regex=True
)

print(f'Cleaned image names - sample:')
print(ring_df['image_name'].head())

# Merge with predictions
ring_df = ring_df.merge(
    predictions_df[['image_name', 'predicted_class', 'prediction_score']],
    on='image_name',
    how='left'
)

# Merge with granular subgroup (cell_type) and patientID
ring_df = ring_df.merge(
    merged_df[['image_name', 'cell_type', 'patientID']],
    on='image_name',
    how='left'
)

print(f'\nring_df shape: {ring_df.shape}')
print(f'\ncell_type distribution:')
print(ring_df['cell_type'].value_counts().reindex(CELL_TYPE_ORDER))

# Per-subgroup analysis groups: Preleukemia subtypes (MDS/CMML/CCUS) and MRD monitoring
# subtypes (negative/unspecific/low-positive/positive), replacing the pooled 2-group comparison.
attr_groups = {ct: ring_df[ring_df['cell_type'] == ct] for ct in CELL_TYPE_ORDER}

print(f'\n✓ Analysis groups (cells / patients, within this occlusion-heatmap subset):')
for ct in CELL_TYPE_ORDER:
    g = attr_groups[ct]
    print(f'  {ct}: {len(g)} cells, {g["patientID"].nunique()} patients')

## 3. Ring-wise Attribution Statistics

In [ ]:
from statsmodels.stats.multitest import multipletests

print('='*80)
print('RING-WISE ATTRIBUTION STATISTICS ACROSS 7 SUBGROUPS (cell-level, SECONDARY)')
print('='*80)

for ct in CELL_TYPE_ORDER:
    g = attr_groups[ct]
    if len(g) == 0:
        print(f'\n{ct}: no cells')
        continue
    print(f'\n{ct} (n={len(g)} cells, {g["patientID"].nunique()} patients):')
    print(f'  Lamina (Ring 0):  Net={g["attr_ring00_mean"].mean():+.4f}')
    print(f'  Center (Ring {N_RINGS-1}): Net={g[f"attr_ring{N_RINGS-1:02d}_mean"].mean():+.4f}')

print(f'\n--- Per-ring Kruskal-Wallis across all 7 subgroups, EVERY ring, BH-FDR across {N_RINGS} rings (cell-level) ---')
cell_stats_rows = []
for i in range(N_RINGS):
    row = {'ring': i, 'region': 'Lamina' if i == 0 else ('Center' if i == N_RINGS - 1 else f'Inner{i}')}
    for kind, prefix in [('positive', 'attr_pos_ring'), ('negative', 'attr_neg_ring'), ('net', 'attr_ring')]:
        col = f'{prefix}{i:02d}_mean'
        samples = [attr_groups[ct][col].dropna() for ct in CELL_TYPE_ORDER if len(attr_groups[ct]) > 0]
        h_stat, p_val = stats.kruskal(*samples)
        row[f'{kind}_kruskal_H'] = h_stat
        row[f'{kind}_kruskal_p'] = p_val
    cell_stats_rows.append(row)

attr_ring_stats = pd.DataFrame(cell_stats_rows)
for kind in ('positive', 'negative', 'net'):
    attr_ring_stats[f'{kind}_kruskal_p_fdr'] = multipletests(attr_ring_stats[f'{kind}_kruskal_p'], method='fdr_bh')[1]
print(attr_ring_stats[['ring', 'region', 'positive_kruskal_p_fdr', 'negative_kruskal_p_fdr', 'net_kruskal_p_fdr']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# ---- PRIMARY: patient-level, EVERY ring, BH-FDR across rings ----
# Unit of analysis = patient (mean attribution per patient per ring), not cell. CCUS, unspecific,
# low-positive, and positive are each a single patient in this subset; MDS, CMML, and negative
# have several patients each and drive most of this test's power.
print('\n' + '='*80)
print('RING-WISE ATTRIBUTION STATISTICS ACROSS 7 SUBGROUPS (patient-level, PRIMARY)')
print('='*80)

attr_val_cols = ([f'attr_pos_ring{i:02d}_mean' for i in range(N_RINGS)] +
                  [f'attr_neg_ring{i:02d}_mean' for i in range(N_RINGS)] +
                  [f'attr_ring{i:02d}_mean' for i in range(N_RINGS)])
patient_attr = ring_df.groupby(['patientID', 'cell_type'], observed=True)[attr_val_cols].mean().reset_index()
attr_patient_groups = {ct: patient_attr[patient_attr['cell_type'] == ct] for ct in CELL_TYPE_ORDER}

n_patients = patient_attr.groupby('cell_type')['patientID'].nunique().reindex(CELL_TYPE_ORDER)
print('Patients per subgroup:')
print(n_patients)
print()

patient_stats_rows = []
for i in range(N_RINGS):
    row = {'ring': i, 'region': 'Lamina' if i == 0 else ('Center' if i == N_RINGS - 1 else f'Inner{i}')}
    for kind, prefix in [('positive', 'attr_pos_ring'), ('negative', 'attr_neg_ring'), ('net', 'attr_ring')]:
        col = f'{prefix}{i:02d}_mean'
        samples = [attr_patient_groups[ct][col].dropna() for ct in CELL_TYPE_ORDER if len(attr_patient_groups[ct]) > 0]
        h_stat, p_val = stats.kruskal(*samples)
        row[f'{kind}_kruskal_H'] = h_stat
        row[f'{kind}_kruskal_p'] = p_val
    patient_stats_rows.append(row)

attr_patient_ring_stats = pd.DataFrame(patient_stats_rows)
for kind in ('positive', 'negative', 'net'):
    attr_patient_ring_stats[f'{kind}_kruskal_p_fdr'] = multipletests(attr_patient_ring_stats[f'{kind}_kruskal_p'], method='fdr_bh')[1]
    attr_patient_ring_stats[f'{kind}_sig'] = attr_patient_ring_stats[f'{kind}_kruskal_p_fdr'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    )
print(attr_patient_ring_stats[['ring', 'region', 'positive_kruskal_p_fdr', 'negative_kruskal_p_fdr', 'net_kruskal_p_fdr', 'net_sig']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('='*80)

attr_ring_stats.to_csv(STATS_DIR / 'attribution_ring_stats_7group_percell.csv', index=False)
attr_patient_ring_stats.to_csv(STATS_DIR / 'attribution_ring_stats_7group_patientlevel.csv', index=False)
ring_df.to_csv(STATS_DIR / 'attribution_ring_features_percell.csv', index=False)
patient_attr.to_csv(STATS_DIR / 'attribution_ring_patient_level.csv', index=False)
print(f'\n✓ Saved: {STATS_DIR / "attribution_ring_stats_7group_percell.csv"}')
print(f'✓ Saved: {STATS_DIR / "attribution_ring_stats_7group_patientlevel.csv"}')
print(f'✓ Saved: {STATS_DIR / "attribution_ring_features_percell.csv"} ({ring_df.shape})')
print(f'✓ Saved: {STATS_DIR / "attribution_ring_patient_level.csv"} ({patient_attr.shape})')

## 4. Ring-wise Attribution Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
x_rings = np.arange(N_RINGS)
active_types = [ct for ct in CELL_TYPE_ORDER if len(attr_groups[ct]) > 0]

# 1. Net attribution profile, all subgroups
for ct in active_types:
    g = attr_groups[ct]
    profile = [g[f'attr_ring{i:02d}_mean'].mean() for i in range(N_RINGS)]
    axes[0, 0].plot(x_rings, profile, 'o-', linewidth=2, markersize=6, label=f'{ct} (n={len(g)})', color=CELL_TYPE_COLORS[ct])
axes[0, 0].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 0].set_xlabel('Ring (0=Lamina → 7=Center)')
axes[0, 0].set_ylabel('Mean Net Attribution')
axes[0, 0].set_title('Net (Pos-Neg) Attribution by Subgroup')
axes[0, 0].set_xticks(x_rings)
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# 2. Positive attribution profile
for ct in active_types:
    g = attr_groups[ct]
    profile = [g[f'attr_pos_ring{i:02d}_mean'].mean() for i in range(N_RINGS)]
    axes[0, 1].plot(x_rings, profile, 'o-', linewidth=2, markersize=6, label=ct, color=CELL_TYPE_COLORS[ct])
axes[0, 1].set_xlabel('Ring (0=Lamina → 7=Center)')
axes[0, 1].set_ylabel('Mean Positive (AML-like) Attribution')
axes[0, 1].set_title('Positive Attribution by Subgroup')
axes[0, 1].set_xticks(x_rings)
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

# 3. Negative attribution profile
for ct in active_types:
    g = attr_groups[ct]
    profile = [g[f'attr_neg_ring{i:02d}_mean'].mean() for i in range(N_RINGS)]
    axes[0, 2].plot(x_rings, profile, 'o-', linewidth=2, markersize=6, label=ct, color=CELL_TYPE_COLORS[ct])
axes[0, 2].set_xlabel('Ring (0=Lamina → 7=Center)')
axes[0, 2].set_ylabel('Mean Negative (Control-like) Attribution')
axes[0, 2].set_title('Negative Attribution by Subgroup')
axes[0, 2].set_xticks(x_rings)
axes[0, 2].legend(fontsize=8)
axes[0, 2].grid(alpha=0.3)

# 4. Lamina net boxplot
box_data = [attr_groups[ct]['attr_ring00_mean'].dropna() for ct in active_types]
bp = axes[1, 0].boxplot(box_data, labels=active_types, patch_artist=True)
for patch, ct in zip(bp['boxes'], active_types):
    patch.set_facecolor(CELL_TYPE_COLORS[ct])
    patch.set_alpha(0.6)
axes[1, 0].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[1, 0].set_ylabel('Net Attribution')
p0 = attr_patient_ring_stats.loc[attr_patient_ring_stats['ring'] == 0, 'net_kruskal_p_fdr'].values[0]
axes[1, 0].set_title(f'Lamina (Ring 0) Net Attribution (patient-level FDR p={p0:.4f})')
axes[1, 0].tick_params(axis='x', rotation=30)
axes[1, 0].grid(alpha=0.3, axis='y')

# 5. Center net boxplot
box_data = [attr_groups[ct][f'attr_ring{N_RINGS-1:02d}_mean'].dropna() for ct in active_types]
bp = axes[1, 1].boxplot(box_data, labels=active_types, patch_artist=True)
for patch, ct in zip(bp['boxes'], active_types):
    patch.set_facecolor(CELL_TYPE_COLORS[ct])
    patch.set_alpha(0.6)
axes[1, 1].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[1, 1].set_ylabel('Net Attribution')
p7 = attr_patient_ring_stats.loc[attr_patient_ring_stats['ring'] == N_RINGS - 1, 'net_kruskal_p_fdr'].values[0]
axes[1, 1].set_title(f'Center (Ring {N_RINGS-1}) Net Attribution (patient-level FDR p={p7:.4f})')
axes[1, 1].tick_params(axis='x', rotation=30)
axes[1, 1].grid(alpha=0.3, axis='y')

# 6. Cell counts per subgroup (context panel)
counts = [len(attr_groups[ct]) for ct in active_types]
axes[1, 2].bar(active_types, counts, color=[CELL_TYPE_COLORS[ct] for ct in active_types], alpha=0.7)
axes[1, 2].set_ylabel('Number of Cells')
axes[1, 2].set_title('Cells per Subgroup (this occlusion subset)')
axes[1, 2].tick_params(axis='x', rotation=30)
axes[1, 2].grid(alpha=0.3, axis='y')

fig.suptitle(f'Ring-wise Occlusion Attribution Across Subgroups (n={len(ring_df)} cells)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / '01_attribution_ring_analysis.png', dpi=200, bbox_inches='tight')
print(f'✓ Saved figure: {FIG_DIR / "01_attribution_ring_analysis.png"}')
plt.show()

print('\n✓ Visualizations complete')

## 5. Ring-wise Raw DAPI Intensity Analysis

The occlusion attribution above shows what the *model* attends to. Here we test whether the *raw DAPI signal itself* differs radially across the Preleukemia subtypes (MDS/CMML/CCUS) and MRD monitoring subtypes (negative/unspecific/low-positive/positive).

This uses precomputed radial-ring DAPI intensity features already extracted directly from the raw single-cell DAPI crops (grayscale, distance-transform "onion-peel" segmentation from cell boundary to center — see `dapi_mask.ipynb`, `extract_cell_features`). 10 concentric rings: Ring 0 = lamina/boundary, Ring 9 = nuclear center.

Unlike the occlusion attribution features (limited to cells with saved heatmaps), these radial DAPI features exist for the full metadata cohort, giving much greater statistical power and no dependency on model attribution.

In [ ]:
# Build ring-wise raw DAPI intensity dataframe (all cells with metadata, not just those with occlusion maps)
N_DAPI_RINGS = 10
DAPI_STAT = 'mean'  # per-ring statistic to analyze

radial_stat_cols = [f'radial_ring{i:02d}_{DAPI_STAT}' for i in range(N_DAPI_RINGS)]
radial_extra_cols = ['radial_monotonicity', 'radial_peak_ring', 'radial_inner_outer_ratio']

dapi_ring_df = metadata_df[['File'] + radial_stat_cols + radial_extra_cols].merge(
    merged_df[['image_name', 'cell_type', 'patientID', 'predicted_class', 'prediction_score']],
    left_on='File', right_on='image_name', how='inner'
)

print(f'dapi_ring_df shape: {dapi_ring_df.shape}')
print(f'\ncell_type distribution:')
print(dapi_ring_df['cell_type'].value_counts().reindex(CELL_TYPE_ORDER))

# Per-subgroup analysis groups (Preleukemia + MRD subtypes)
dapi_groups = {ct: dapi_ring_df[dapi_ring_df['cell_type'] == ct] for ct in CELL_TYPE_ORDER}

print(f'\n✓ Analysis groups (cells / patients):')
for ct in CELL_TYPE_ORDER:
    g = dapi_groups[ct]
    print(f'  {ct}: {len(g)} cells, {g["patientID"].nunique()} patients')

In [ ]:
from statsmodels.stats.multitest import multipletests

print('='*80)
print('RING-WISE RAW DAPI INTENSITY STATISTICS ACROSS 7 SUBGROUPS (cell-level, SECONDARY)')
print('='*80)

dapi_cell_stats_rows = []
for i in range(N_DAPI_RINGS):
    col = f'radial_ring{i:02d}_{DAPI_STAT}'
    samples = [dapi_groups[ct][col].dropna() for ct in CELL_TYPE_ORDER if len(dapi_groups[ct]) > 0]
    h_stat, p_val = stats.kruskal(*samples)
    dapi_cell_stats_rows.append({
        'ring': i, 'region': 'Lamina' if i == 0 else ('Center' if i == N_DAPI_RINGS - 1 else f'Inner{i}'),
        'kruskal_H': h_stat, 'kruskal_p': p_val,
    })
dapi_ring_stats = pd.DataFrame(dapi_cell_stats_rows)
dapi_ring_stats['kruskal_p_fdr'] = multipletests(dapi_ring_stats['kruskal_p'], method='fdr_bh')[1]
print(dapi_ring_stats.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# ---- PRIMARY: patient-level, EVERY ring, BH-FDR across rings ----
print('\n' + '='*80)
print('RING-WISE RAW DAPI INTENSITY STATISTICS ACROSS 7 SUBGROUPS (patient-level, PRIMARY)')
print('='*80)

patient_ring = dapi_ring_df.groupby(['patientID', 'cell_type'], observed=True)[radial_stat_cols].mean().reset_index()
dapi_patient_groups = {ct: patient_ring[patient_ring['cell_type'] == ct] for ct in CELL_TYPE_ORDER}

n_patients = patient_ring.groupby('cell_type')['patientID'].nunique().reindex(CELL_TYPE_ORDER)
print('Patients per subgroup:')
print(n_patients)
print()

dapi_patient_stats_rows = []
for i in range(N_DAPI_RINGS):
    col = f'radial_ring{i:02d}_{DAPI_STAT}'
    samples = [dapi_patient_groups[ct][col].dropna() for ct in CELL_TYPE_ORDER if len(dapi_patient_groups[ct]) > 0]
    h_stat, p_val = stats.kruskal(*samples)
    dapi_patient_stats_rows.append({
        'ring': i, 'region': 'Lamina' if i == 0 else ('Center' if i == N_DAPI_RINGS - 1 else f'Inner{i}'),
        'kruskal_H': h_stat, 'kruskal_p': p_val,
    })
dapi_patient_ring_stats = pd.DataFrame(dapi_patient_stats_rows)
dapi_patient_ring_stats['kruskal_p_fdr'] = multipletests(dapi_patient_ring_stats['kruskal_p'], method='fdr_bh')[1]
dapi_patient_ring_stats['sig'] = dapi_patient_ring_stats['kruskal_p_fdr'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
)
print(dapi_patient_ring_stats.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('='*80)

dapi_ring_stats.to_csv(STATS_DIR / 'dapi_radial_ring_stats_7group_percell.csv', index=False)
dapi_patient_ring_stats.to_csv(STATS_DIR / 'dapi_radial_ring_stats_7group_patientlevel.csv', index=False)
dapi_ring_df.to_csv(STATS_DIR / 'dapi_radial_ring_features_percell.csv', index=False)
patient_ring.to_csv(STATS_DIR / 'dapi_radial_ring_patient_level.csv', index=False)
print(f'\n✓ Saved: {STATS_DIR / "dapi_radial_ring_stats_7group_percell.csv"}')
print(f'✓ Saved: {STATS_DIR / "dapi_radial_ring_stats_7group_patientlevel.csv"}')
print(f'✓ Saved: {STATS_DIR / "dapi_radial_ring_features_percell.csv"} ({dapi_ring_df.shape})')
print(f'✓ Saved: {STATS_DIR / "dapi_radial_ring_patient_level.csv"} ({patient_ring.shape})')

> **⚠️ Caveat — patient imbalance across the 7 subgroups.** CCUS, unspecific, low-positive, and positive are each represented by a single patient; MDS, CMML, and negative have several patients each and drive most of the patient-level test's power. Any significant ring in the omnibus test could be driven by MDS/CMML/negative differences rather than by the single-patient subgroups — the patient-level result (below) is the primary one, but single-patient subgroups still cannot be distinguished from a patient/staining-batch effect. **Treat any finding involving CCUS, unspecific, low-positive, or positive as hypothesis-generating, not confirmatory, until more patients are added for those subgroups.**

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
x_rings = np.arange(N_DAPI_RINGS)
active_types = [ct for ct in CELL_TYPE_ORDER if len(dapi_groups[ct]) > 0]

# 1. Raw DAPI intensity profile, all subgroups
for ct in active_types:
    g = dapi_groups[ct]
    profile = [g[f'radial_ring{i:02d}_{DAPI_STAT}'].mean() for i in range(N_DAPI_RINGS)]
    axes[0, 0].plot(x_rings, profile, 'o-', linewidth=2, markersize=6, label=f'{ct} (n={len(g)})', color=CELL_TYPE_COLORS[ct])
axes[0, 0].set_xlabel('Ring (0=Lamina → 9=Center)')
axes[0, 0].set_ylabel('Mean DAPI Intensity (grayscale)')
axes[0, 0].set_title('Raw DAPI Intensity by Subgroup')
axes[0, 0].set_xticks(x_rings)
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# 2. Radial monotonicity by subgroup
box_data = [dapi_groups[ct]['radial_monotonicity'].dropna() for ct in active_types]
bp = axes[0, 1].boxplot(box_data, labels=active_types, patch_artist=True)
for patch, ct in zip(bp['boxes'], active_types):
    patch.set_facecolor(CELL_TYPE_COLORS[ct]); patch.set_alpha(0.6)
axes[0, 1].set_ylabel('Radial Monotonicity (Spearman ρ)')
axes[0, 1].set_title('DAPI Radial Gradient Consistency')
axes[0, 1].tick_params(axis='x', rotation=30)
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. Inner/outer chromatin ratio by subgroup
box_data = [dapi_groups[ct]['radial_inner_outer_ratio'].dropna() for ct in active_types]
bp = axes[0, 2].boxplot(box_data, labels=active_types, patch_artist=True)
for patch, ct in zip(bp['boxes'], active_types):
    patch.set_facecolor(CELL_TYPE_COLORS[ct]); patch.set_alpha(0.6)
axes[0, 2].set_ylabel('Inner/Outer DAPI Ratio')
axes[0, 2].set_title('Nuclear Chromatin Concentration')
axes[0, 2].tick_params(axis='x', rotation=30)
axes[0, 2].grid(alpha=0.3, axis='y')

# 4. Lamina (ring 0) boxplot - patient-level FDR p in title
box_data = [dapi_groups[ct]['radial_ring00_mean'].dropna() for ct in active_types]
bp = axes[1, 0].boxplot(box_data, labels=active_types, patch_artist=True)
for patch, ct in zip(bp['boxes'], active_types):
    patch.set_facecolor(CELL_TYPE_COLORS[ct]); patch.set_alpha(0.6)
axes[1, 0].set_ylabel('DAPI Intensity')
p0 = dapi_patient_ring_stats.loc[dapi_patient_ring_stats['ring'] == 0, 'kruskal_p_fdr'].values[0]
axes[1, 0].set_title(f'Lamina (Ring 0) DAPI (patient-level FDR p={p0:.4f})')
axes[1, 0].tick_params(axis='x', rotation=30)
axes[1, 0].grid(alpha=0.3, axis='y')

# 5. Center (ring 9) boxplot - patient-level FDR p in title
box_data = [dapi_groups[ct][f'radial_ring{N_DAPI_RINGS-1:02d}_mean'].dropna() for ct in active_types]
bp = axes[1, 1].boxplot(box_data, labels=active_types, patch_artist=True)
for patch, ct in zip(bp['boxes'], active_types):
    patch.set_facecolor(CELL_TYPE_COLORS[ct]); patch.set_alpha(0.6)
axes[1, 1].set_ylabel('DAPI Intensity')
p9 = dapi_patient_ring_stats.loc[dapi_patient_ring_stats['ring'] == N_DAPI_RINGS - 1, 'kruskal_p_fdr'].values[0]
axes[1, 1].set_title(f'Center (Ring {N_DAPI_RINGS-1}) DAPI (patient-level FDR p={p9:.4f})')
axes[1, 1].tick_params(axis='x', rotation=30)
axes[1, 1].grid(alpha=0.3, axis='y')

# 6. Cell counts per subgroup (context panel)
counts = [len(dapi_groups[ct]) for ct in active_types]
axes[1, 2].bar(active_types, counts, color=[CELL_TYPE_COLORS[ct] for ct in active_types], alpha=0.7)
axes[1, 2].set_ylabel('Number of Cells')
axes[1, 2].set_title('Cells per Subgroup')
axes[1, 2].tick_params(axis='x', rotation=30)
axes[1, 2].grid(alpha=0.3, axis='y')

fig.suptitle(f'Ring-wise Raw DAPI Intensity Across Subgroups (n={len(dapi_ring_df)} cells)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / '02_dapi_radial_ring_analysis.png', dpi=200, bbox_inches='tight')
print(f'✓ Saved figure: {FIG_DIR / "02_dapi_radial_ring_analysis.png"}')
plt.show()

print('\n✓ Visualizations complete')

## Summary

All three ring-wise analyses below are now run at the level of the **7 clinical subgroups**
(Preleukemia: MDS, CMML, CCUS; MRD monitoring: negative, unspecific, low-positive, positive —
excluding 61 "AML _post-CMML" cells), each with a **cell-level test (secondary, every ring,
BH-FDR corrected)** and a **patient-level test (primary, every ring, BH-FDR corrected)**, since
cell-level pseudo-replication was inflating significance throughout the earlier pooled
MRD-vs-Preleukemia version of this notebook.

**Ring-wise occlusion attribution** (n=2,887 cells — note **CMML and CCUS have zero occlusion
heatmaps on disk**, confirmed not a matching bug, so only 5 of 7 subgroups are testable here):
- Cell-level (secondary): every ring significant for positive/negative/net attribution (FDR p<0.01).
- **Patient-level (primary): nothing is significant at any ring** (net attribution FDR p=0.54–0.58 across all 8 rings). Only MDS (11 patients) and negative (14 patients) contribute meaningfully; unspecific/low-positive/positive are single patients each.
- **This nullifies the earlier pooled-group finding** (that Rings 2–7 were significant for MRD vs. Preleukemia): once split into actual clinical subgroups and tested with patients as the unit of analysis, that signal does not survive.

**Ring-wise raw DAPI intensity** (n=5,892 cells, all 7 subgroups present):
- Cell-level (secondary): every ring significant (FDR p<0.0001) — driven by the very large n and 7-way comparison.
- **Patient-level (primary): nothing is significant at any ring** (FDR p=0.27–0.75 across all 10 rings), despite MDS (12), CMML (3), and negative (14) patients being available.

**Ring-wise DNA density/content** (n=5,892 cells, all 7 subgroups present):
- Cell-level (secondary): every ring and the whole-cell total are significant (FDR/omnibus p<0.0001).
- **Patient-level (primary, MDS/CMML/negative — the 3 subgroups with enough patients): nothing is significant at any ring** (FDR p=0.54–0.84).

**Overall conclusion**: across all three independent readouts (model attribution, raw DAPI intensity, DNA density/content), the pattern is the same — **strong cell-level "significance" that completely disappears once analysis is done at the patient level with proper multiple-comparison correction.** This is a much more rigorous and much less encouraging result than the earlier pooled 2-group version of this notebook suggested: none of the ring-wise features examined here currently show robust evidence of differing between Preleukemia and MRD subgroups, or among the Preleukemia/MRD subgroups themselves. The cell-level "signal" documented throughout this notebook is very likely dominated by patient identity (individual staining/imaging characteristics) rather than by disease- or MRD-status biology. **The single-patient subgroups (CCUS, unspecific, low-positive, positive) are the binding constraint — more patients per subgroup, especially for those four, are required before any of these ring-wise readouts can be trusted as a biological finding.**

## 5. Ring-wise DNA Density and Total DNA Content (Raw Images)

Using the raw DAPI cell-crop PNGs in `data/human_data/`, segment each nucleus, decompose it into
the same 8 concentric rings (0=periphery/lamina, 7=center) used in the occlusion analysis, and
measure per-ring **DNA density** (mean pixel intensity) and **DNA content** (summed intensity),
plus whole-cell total DNA content. Grouped by the 7 subgroups: Preleukemia disease types
(MDS/CMML/CCUS) and MRD monitoring subtypes (negative/unspecific/low-positive/positive).

In [ ]:
# Extract ring-wise DNA density/content from raw images for every cell
N_RINGS = 8
dna_data = []
failed = 0

for idx, row in merged_df.iterrows():
    if (idx + 1) % 500 == 0:
        print(f'  {idx + 1}/{len(merged_df)} processed, {len(dna_data)} extracted')

    name = row['image_name']
    img_path = IMAGE_DIR / f'{name}.png'
    if not img_path.exists():
        failed += 1
        continue

    try:
        img = Image.open(img_path).convert('L')
        arr = np.array(img).astype(np.float64)

        mask = extract_cell_mask_bg(arr)
        if mask.sum() < 10:
            failed += 1
            continue

        rmap = ring_map(mask, n_rings=N_RINGS)
        row_data = {'image_name': name}

        for r in range(N_RINGS):
            ring_mask = rmap == r
            if ring_mask.sum() > 0:
                vals = arr[ring_mask]
                row_data[f'dna_density_ring{r:02d}'] = float(vals.mean())
                row_data[f'dna_total_ring{r:02d}'] = float(vals.sum())
                row_data[f'ring{r:02d}_n_px'] = int(ring_mask.sum())

        row_data['total_dna_content'] = float(arr[mask].sum())
        row_data['mean_dna_density'] = float(arr[mask].mean())
        row_data['n_nucleus_px'] = int(mask.sum())

        dna_data.append(row_data)
    except Exception as e:
        failed += 1
        if failed <= 5:
            print(f'  FAILED on {name}: {type(e).__name__}: {e}')
        continue

print(f'\n✓ Extracted DNA features from {len(dna_data)} cells ({failed} failed/skipped)')

In [ ]:
# Build dna_df and merge with cell_type
dna_df = pd.DataFrame(dna_data)
dna_df = dna_df.merge(merged_df[['image_name', 'cell_type', 'patientID']], on='image_name', how='left')

print(f'dna_df shape: {dna_df.shape}')
print(f'\ncell_type distribution in dna_df:')
print(dna_df['cell_type'].value_counts().reindex(CELL_TYPE_ORDER))

dna_groups = {ct: dna_df[dna_df['cell_type'] == ct] for ct in CELL_TYPE_ORDER}


### DNA Density/Content Statistics

In [ ]:
from statsmodels.stats.multitest import multipletests

print('='*80)
print('RING-WISE DNA DENSITY & CONTENT: STATISTICS ACROSS 7 SUBGROUPS (cell-level)')
print('='*80)

group_summary_rows = []
for ct in CELL_TYPE_ORDER:
    g = dna_groups[ct]
    print(f'\n{ct} (n={len(g)}):')
    print(f'  Total DNA content (whole cell): {g["total_dna_content"].mean():.1f} ± {g["total_dna_content"].std():.1f}')
    print(f'  Lamina (Ring 0) density:  {g["dna_density_ring00"].mean():.2f} ± {g["dna_density_ring00"].std():.2f}')
    print(f'  Center (Ring 7) density:  {g["dna_density_ring07"].mean():.2f} ± {g["dna_density_ring07"].std():.2f}')
    row = {'cell_type': ct, 'n': len(g),
           'total_dna_content_mean': g['total_dna_content'].mean(), 'total_dna_content_std': g['total_dna_content'].std(),
           'n_nucleus_px_mean': g['n_nucleus_px'].mean(), 'n_nucleus_px_std': g['n_nucleus_px'].std()}
    for r in range(N_RINGS):
        row[f'dna_density_ring{r:02d}_mean'] = g[f'dna_density_ring{r:02d}'].mean()
        row[f'dna_density_ring{r:02d}_std'] = g[f'dna_density_ring{r:02d}'].std()
        row[f'dna_total_ring{r:02d}_mean'] = g[f'dna_total_ring{r:02d}'].mean()
        row[f'dna_total_ring{r:02d}_std'] = g[f'dna_total_ring{r:02d}'].std()
    group_summary_rows.append(row)
dna_group_summary = pd.DataFrame(group_summary_rows)

print(f'\n--- Kruskal-Wallis across all 7 subgroups (omnibus, single test - no FDR needed) ---')
kw_scalar_rows = []
for label_, col in [('Total DNA content', 'total_dna_content'),
                     ('Lamina density (Ring 0)', 'dna_density_ring00'),
                     ('Center density (Ring 7)', 'dna_density_ring07')]:
    samples = [dna_groups[ct][col].dropna() for ct in CELL_TYPE_ORDER if len(dna_groups[ct]) > 0]
    h_stat, p_val = stats.kruskal(*samples)
    print(f'  {label_}: H={h_stat:.2f}, p={p_val:.6f}' + (' ★ SIGNIFICANT' if p_val < 0.05 else ''))
    kw_scalar_rows.append({'metric': label_, 'column': col, 'kruskal_H': h_stat, 'kruskal_p': p_val})

print(f'\n--- Per-ring Kruskal-Wallis across all 7 subgroups, EVERY ring, BH-FDR across the {N_RINGS} rings ---')
kw_ring_rows = []
for r in range(N_RINGS):
    dens_samples = [dna_groups[ct][f'dna_density_ring{r:02d}'].dropna() for ct in CELL_TYPE_ORDER if len(dna_groups[ct]) > 0]
    tot_samples = [dna_groups[ct][f'dna_total_ring{r:02d}'].dropna() for ct in CELL_TYPE_ORDER if len(dna_groups[ct]) > 0]
    h_dens, p_dens = stats.kruskal(*dens_samples)
    h_tot, p_tot = stats.kruskal(*tot_samples)
    kw_ring_rows.append({'ring': r, 'density_kruskal_H': h_dens, 'density_kruskal_p': p_dens,
                          'total_kruskal_H': h_tot, 'total_kruskal_p': p_tot})
dna_kw_ring = pd.DataFrame(kw_ring_rows)
dna_kw_ring['density_kruskal_p_fdr'] = multipletests(dna_kw_ring['density_kruskal_p'], method='fdr_bh')[1]
dna_kw_ring['total_kruskal_p_fdr'] = multipletests(dna_kw_ring['total_kruskal_p'], method='fdr_bh')[1]
dna_kw_ring['density_sig'] = dna_kw_ring['density_kruskal_p_fdr'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
)
print(dna_kw_ring.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
print('\n⚠️ This is a 7-SUBGROUP CELL-LEVEL omnibus test - see the patient-level check below: 4 of')
print('   these 6 groups (CCUS, unspecific, low-positive, positive) are each a SINGLE patient,')
print('   so this result is confounded with patient/imaging-batch identity for those groups.')
print('='*80)

# Save stats and per-cell feature table
dna_kw_scalar = pd.DataFrame(kw_scalar_rows)

dna_group_summary.to_csv(STATS_DIR / 'dna_density_group_summary.csv', index=False)
dna_kw_scalar.to_csv(STATS_DIR / 'dna_density_kruskal_scalar.csv', index=False)
dna_kw_ring.to_csv(STATS_DIR / 'dna_density_kruskal_perring.csv', index=False)
dna_df.to_csv(STATS_DIR / 'dna_density_features_percell.csv', index=False)
print(f'\n✓ Saved: {STATS_DIR / "dna_density_group_summary.csv"} ({dna_group_summary.shape})')
print(f'✓ Saved: {STATS_DIR / "dna_density_kruskal_scalar.csv"} ({dna_kw_scalar.shape})')
print(f'✓ Saved: {STATS_DIR / "dna_density_kruskal_perring.csv"} ({dna_kw_ring.shape})')
print(f'✓ Saved: {STATS_DIR / "dna_density_features_percell.csv"} ({dna_df.shape})')

> **⚠️ Caveat — patient imbalance across the 7 subgroups.** CCUS, unspecific, low-positive, and positive are each represented by exactly **1 patient**; MDS, CMML, and negative have enough patients for a meaningful patient-level test. The cell-level 7-group comparison above is therefore only interpretable at face value for MDS/CMML/negative — for the other 4 groups it cannot distinguish a true disease/MRD-status effect from a patient/staining-batch effect. The cell below runs the patient-level comparison across the 3 groups that actually have enough patients (every ring, BH-FDR corrected).

In [ ]:
from statsmodels.stats.multitest import multipletests

print('='*80)
print('PATIENT-LEVEL DNA DENSITY CHECK')
print('='*80)

n_patients_per_type = dna_df.groupby('cell_type')['patientID'].nunique().reindex(CELL_TYPE_ORDER)
print('Patients per cell type:')
print(n_patients_per_type)
print('\nOnly MDS, CMML, and negative have >1 patient per group; those 3 are tested below.\n')

dna_ring_cols = [f'dna_density_ring{r:02d}' for r in range(N_RINGS)]
patient_dna = dna_df.groupby(['patientID', 'cell_type'], observed=True)[dna_ring_cols + ['total_dna_content']].mean().reset_index()

WELL_POWERED_TYPES = ['MDS', 'CMML', 'negative']
patient_dna_groups = {ct: patient_dna[patient_dna['cell_type'] == ct] for ct in WELL_POWERED_TYPES}
for ct in WELL_POWERED_TYPES:
    print(f'{ct} patients: {len(patient_dna_groups[ct])}')
print()

dna_patient_stats_rows = []
for r in range(N_RINGS):
    col = f'dna_density_ring{r:02d}'
    samples = [patient_dna_groups[ct][col].dropna() for ct in WELL_POWERED_TYPES if len(patient_dna_groups[ct]) > 0]
    h_stat, p_val = stats.kruskal(*samples) if len(samples) >= 2 else (np.nan, np.nan)
    row = {
        'ring': r,
        'region': 'Lamina' if r == 0 else ('Center' if r == N_RINGS - 1 else f'Inner{r}'),
        'kruskal_H': h_stat, 'kruskal_p': p_val,
    }
    for ct in WELL_POWERED_TYPES:
        row[f'{ct}_patient_mean'] = patient_dna_groups[ct][col].mean()
    dna_patient_stats_rows.append(row)

dna_patient_ring_stats = pd.DataFrame(dna_patient_stats_rows)
valid = dna_patient_ring_stats['kruskal_p'].notna()
dna_patient_ring_stats['kruskal_p_fdr'] = np.nan
dna_patient_ring_stats.loc[valid, 'kruskal_p_fdr'] = multipletests(
    dna_patient_ring_stats.loc[valid, 'kruskal_p'], method='fdr_bh'
)[1]
dna_patient_ring_stats['sig'] = dna_patient_ring_stats['kruskal_p_fdr'].apply(
    lambda p: ('***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))) if pd.notna(p) else 'n/a'
)
print(dna_patient_ring_stats.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('='*80)

dna_patient_ring_stats.to_csv(STATS_DIR / 'dna_density_patientlevel_MDS_CMML_negative.csv', index=False)
patient_dna.to_csv(STATS_DIR / 'dna_density_patient_level_allgroups.csv', index=False)
print(f'\n✓ Saved: {STATS_DIR / "dna_density_patientlevel_MDS_CMML_negative.csv"} ({dna_patient_ring_stats.shape})')
print(f'✓ Saved: {STATS_DIR / "dna_density_patient_level_allgroups.csv"} ({patient_dna.shape})')

### DNA Density/Content Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
x_rings = np.arange(N_RINGS)

# 1. Ring density profile, all 6 groups
for ct in CELL_TYPE_ORDER:
    g = dna_groups[ct]
    profile = [g[f'dna_density_ring{r:02d}'].mean() for r in range(N_RINGS)]
    axes[0, 0].plot(x_rings, profile, 'o-', linewidth=2, markersize=6, label=f'{ct} (n={len(g)})', color=CELL_TYPE_COLORS[ct])
axes[0, 0].set_xlabel('Ring (0=Lamina → 7=Center)')
axes[0, 0].set_ylabel('Mean DNA Density (intensity)')
axes[0, 0].set_title('DNA Density Profile by Cell Type')
axes[0, 0].set_xticks(x_rings)
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# 2. Ring total DNA content profile
for ct in CELL_TYPE_ORDER:
    g = dna_groups[ct]
    profile = [g[f'dna_total_ring{r:02d}'].mean() for r in range(N_RINGS)]
    axes[0, 1].plot(x_rings, profile, 'o-', linewidth=2, markersize=6, label=ct, color=CELL_TYPE_COLORS[ct])
axes[0, 1].set_xlabel('Ring (0=Lamina → 7=Center)')
axes[0, 1].set_ylabel('Mean Total DNA in Ring')
axes[0, 1].set_title('DNA Content Profile by Cell Type')
axes[0, 1].set_xticks(x_rings)
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

# 3. Whole-cell total DNA content boxplot
box_data = [dna_groups[ct]['total_dna_content'].dropna() for ct in CELL_TYPE_ORDER]
bp = axes[0, 2].boxplot(box_data, labels=CELL_TYPE_ORDER, patch_artist=True)
for patch, ct in zip(bp['boxes'], CELL_TYPE_ORDER):
    patch.set_facecolor(CELL_TYPE_COLORS[ct])
    patch.set_alpha(0.6)
axes[0, 2].set_ylabel('Total DNA Content (whole cell)')
axes[0, 2].set_title('Whole-Cell Total DNA Content')
axes[0, 2].tick_params(axis='x', rotation=30)
axes[0, 2].grid(alpha=0.3, axis='y')

# 4. Lamina density boxplot
box_data = [dna_groups[ct]['dna_density_ring00'].dropna() for ct in CELL_TYPE_ORDER]
bp = axes[1, 0].boxplot(box_data, labels=CELL_TYPE_ORDER, patch_artist=True)
for patch, ct in zip(bp['boxes'], CELL_TYPE_ORDER):
    patch.set_facecolor(CELL_TYPE_COLORS[ct])
    patch.set_alpha(0.6)
axes[1, 0].set_ylabel('DNA Density')
axes[1, 0].set_title('Lamina (Ring 0) DNA Density')
axes[1, 0].tick_params(axis='x', rotation=30)
axes[1, 0].grid(alpha=0.3, axis='y')

# 5. Center density boxplot
box_data = [dna_groups[ct]['dna_density_ring07'].dropna() for ct in CELL_TYPE_ORDER]
bp = axes[1, 1].boxplot(box_data, labels=CELL_TYPE_ORDER, patch_artist=True)
for patch, ct in zip(bp['boxes'], CELL_TYPE_ORDER):
    patch.set_facecolor(CELL_TYPE_COLORS[ct])
    patch.set_alpha(0.6)
axes[1, 1].set_ylabel('DNA Density')
axes[1, 1].set_title('Center (Ring 7) DNA Density')
axes[1, 1].tick_params(axis='x', rotation=30)
axes[1, 1].grid(alpha=0.3, axis='y')

# 6. Nucleus pixel count (size proxy) boxplot
box_data = [dna_groups[ct]['n_nucleus_px'].dropna() for ct in CELL_TYPE_ORDER]
bp = axes[1, 2].boxplot(box_data, labels=CELL_TYPE_ORDER, patch_artist=True)
for patch, ct in zip(bp['boxes'], CELL_TYPE_ORDER):
    patch.set_facecolor(CELL_TYPE_COLORS[ct])
    patch.set_alpha(0.6)
axes[1, 2].set_ylabel('Nucleus Area (pixels)')
axes[1, 2].set_title('Nucleus Size by Cell Type')
axes[1, 2].tick_params(axis='x', rotation=30)
axes[1, 2].grid(alpha=0.3, axis='y')

fig.suptitle('Ring-wise DNA Density & Content: MRD Types vs Preleukemia Types', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR / '03_dna_density_content_by_celltype.png', dpi=200, bbox_inches='tight')
print(f'✓ Saved figure: {FIG_DIR / "03_dna_density_content_by_celltype.png"}')
plt.show()

print('✓ DNA density/content visualizations complete')

### Summary: DNA Density/Content Findings

Ring-wise DNA density and total content analysis complete across all 7 subgroups
(MDS, CMML, CCUS Preleukemia subtypes; negative, unspecific, low-positive, positive MRD
subtypes). Kruskal-Wallis tests identify which rings and which whole-cell metrics differ
significantly across groups, independent of the occlusion-based attribution analysis above —
this uses only raw pixel intensity from the DAPI images, giving a direct structural/chromatin-
density readout to compare against the model's attribution-based findings.

**Cell-level (7-group, every ring, BH-FDR corrected)**: every ring and the whole-cell total
are significant. But 4 of the 7 groups (CCUS, unspecific, low-positive, positive) are each a
single patient, so this result is confounded with patient identity for those groups.

**Patient-level (MDS / CMML / negative — the 3 subgroups with enough patients, every ring,
BH-FDR corrected)**: no ring reaches significance (all FDR p≥0.54). Once patient identity and
multiple comparisons are accounted for, none of the apparent cell-level differences between
these three subgroups are supported.

## 6. Example Cell Masks and Ring Overlays by Group

Visual sanity check of the segmentation used above: for one representative cell per group (closest to that group's median total DNA content), show the raw DAPI crop, the Otsu segmentation mask, and the resulting concentric-ring decomposition (0=lamina → 7=center).

In [ ]:
import matplotlib.patches as mpatches

fig, axes = plt.subplots(len(CELL_TYPE_ORDER), 3, figsize=(9, 3 * len(CELL_TYPE_ORDER)))

example_records = []
for row_idx, ct in enumerate(CELL_TYPE_ORDER):
    g = dna_groups[ct]
    med = g['total_dna_content'].median()
    example_idx = (g['total_dna_content'] - med).abs().idxmin()
    example = g.loc[example_idx]
    example_name = example['image_name']

    img = Image.open(IMAGE_DIR / f'{example_name}.png').convert('L')
    arr = np.array(img).astype(np.float64)

    mask = extract_cell_mask_bg(arr)
    rmap = ring_map(mask, n_rings=N_RINGS)

    # Panel 1: raw image
    axes[row_idx, 0].imshow(arr, cmap='gray')
    axes[row_idx, 0].set_title(f'{ct} (n={len(g)}): {example_name}\nRaw DAPI', fontsize=8)
    axes[row_idx, 0].axis('off')

    # Panel 2: segmentation mask overlay
    axes[row_idx, 1].imshow(arr, cmap='gray')
    axes[row_idx, 1].imshow(np.ma.masked_where(~mask, np.ones_like(arr)), cmap='Reds', alpha=0.45, vmin=0, vmax=1)
    axes[row_idx, 1].set_title('Segmentation Mask (bg threshold)', fontsize=8)
    axes[row_idx, 1].axis('off')

    # Panel 3: ring decomposition overlay
    ring_display = np.ma.masked_where(rmap < 0, rmap)
    axes[row_idx, 2].imshow(arr, cmap='gray')
    axes[row_idx, 2].imshow(ring_display, cmap='viridis', alpha=0.55, vmin=0, vmax=N_RINGS - 1)
    axes[row_idx, 2].set_title(f'{N_RINGS} Rings (0=Lamina → {N_RINGS-1}=Center)', fontsize=8)
    axes[row_idx, 2].axis('off')

    example_records.append({
        'cell_type': ct, 'image_name': example_name, 'n_group': len(g),
        'total_dna_content': example['total_dna_content'], 'n_nucleus_px': example['n_nucleus_px'],
    })

# Shared ring colorbar / legend
ring_cmap = plt.cm.viridis
patches = [mpatches.Patch(color=ring_cmap(r / (N_RINGS - 1)), label=f'Ring {r}') for r in range(N_RINGS)]
fig.legend(handles=patches, loc='lower center', ncol=N_RINGS, bbox_to_anchor=(0.5, -0.01), fontsize=8)

#fig.suptitle('Example Cells: Segmentation Mask and Ring Decomposition by Group', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.02, 1, 1])
fig.savefig(FIG_DIR / '04_example_masks_rings_by_group.png', dpi=200, bbox_inches='tight')
print(f'✓ Saved figure: {FIG_DIR / "04_example_masks_rings_by_group.png"}')
plt.show()

example_cells_df = pd.DataFrame(example_records)
example_cells_df.to_csv(STATS_DIR / 'example_cells_by_group.csv', index=False)
print(f'\n✓ Saved: {STATS_DIR / "example_cells_by_group.csv"}')
print(example_cells_df.to_string(index=False))